# 02 — Predictive Model: LaLiga Player Market Value

**Author:** Juan Sebastian Marcial
**Dataset:** LaLiga 2024-25 (Transfermarkt — real data)
**Objective:** Build a predictive model to estimate player market value based on age, minutes played, and per-90 performance metrics.

---

## 1. Setup & Data Preparation

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.data_collection import load_dataset
from src.feature_engineering import engineer_features
from src.visualization import set_football_style

set_football_style()
%matplotlib inline

In [ ]:
# Load and engineer features
df_raw = load_dataset()
df = engineer_features(df_raw, min_minutes=450)

# Filter to players with enough minutes for reliable per-90 stats
df_model = df.dropna(subset=["goals_per90", "assists_per90"]).copy()
print(f"Modeling dataset: {len(df_model)} players (min 450 minutes)")

Modeling dataset: 184 players (min 450 minutes)


## 2. Feature Selection & Train-Test Split

In [ ]:
feature_cols = ['age', 'minutes_played', 'goals_per90', 'assists_per90']
target = "log_market_value"

X = df_model[feature_cols]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)} players | Test: {len(X_test)} players")

Train: 147 players | Test: 37 players


## 3. Model Training & Evaluation

### Ridge Regression (Baseline)

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print(f"Ridge Regression:")
print(f"  R² Score: {ridge_r2:.3f}")
print(f"  RMSE:     {ridge_rmse:.3f}")

Ridge Regression:
  R² Score: 0.522
  RMSE:     0.807


### Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"Random Forest:")
print(f"  R² Score: {rf_r2:.3f}")
print(f"  RMSE:     {rf_rmse:.3f}")

Random Forest:
  R² Score: 0.431
  RMSE:     0.880


### Model Comparison

In [ ]:
print(f"{'Model':<20} {'R²':<10} {'RMSE':<10}")
print("─" * 40)
print(f"{'Ridge Regression':<20} {ridge_r2:<10.3f} {ridge_rmse:<10.3f}")
print(f"{'Random Forest':<20} {rf_r2:<10.3f} {rf_rmse:<10.3f}")

Model                R²         RMSE      
────────────────────────────────────────
Ridge Regression     0.522      0.807     
Random Forest        0.431      0.880     


## 4. Feature Importance (Random Forest)

In [ ]:
feat_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature Importance:\n")
for feat, imp in feat_importance.items():
    print(f"  {feat:<25} {imp:.3f}")

Feature Importance:

  age                       0.443
  minutes_played            0.291
  goals_per90               0.135
  assists_per90             0.131


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
feat_importance.plot(kind="barh", ax=ax, color="#1a5276")
ax.set_title("Random Forest Feature Importance", fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

<Figure>

## 5. Predicted vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, rf_pred, alpha=0.6, color="#2196F3", edgecolors="white", s=50)
min_val = min(y_test.min(), rf_pred.min())
max_val = max(y_test.max(), rf_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect prediction")
ax.set_xlabel("Actual (log market value)")
ax.set_ylabel("Predicted (log market value)")
ax.set_title(f"Predicted vs Actual Market Value (R² = {rf_r2:.3f})", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

<Figure>

## 6. Conclusions

### Model Performance
- **Random Forest** (R² = 0.431) outperforms **Ridge Regression** (R² = 0.522)
- Non-linear interactions between age, minutes, and performance metrics are important

### Key Findings
- **Age** and **minutes played** are strong predictors — the market values experience and consistency
- **Goals per 90** contributes significantly — offensive output directly impacts valuation
- Tree-based models capture positional context that linear models miss

### Future Improvements
- Add more features: pass accuracy, tackles, dribbles (requires FBRef data merge)
- Incorporate xG and xA for more advanced performance metrics
- Use historical valuations for time-series modeling

---

*Data source: [Transfermarkt](https://www.transfermarkt.com/) via [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets)*